## Prepare data
Load a multirep session from a `session.pkl` file, session directory, or `.tar.gz` archive.

In [2]:
import sys, importlib.util
from pathlib import Path

# VS Code injects __vsc_ipynb_file__ with the notebook's absolute path.
_nb = globals().get("__vsc_ipynb_file__") or ""
if not _nb:
    raise RuntimeError("Select the project venv kernel: .venv/bin/python")

REPO_ROOT = Path(_nb).resolve().parent.parent  # analysis/ -> repo root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Import directly — skips analysis/__init__.py (avoids eager matplotlib/torch load)
def _load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

_loader = _load_module("multirep_loader", REPO_ROOT / "analysis" / "multirep_loader.py")
load_session              = _loader.load_session
load_session_from_tarball = _loader.load_session_from_tarball

# Point to a session directory, session.pkl, or .tar.gz archive.
# Set to None to auto-select the latest .tar.gz in multirepData/.
SESSION_PATH = REPO_ROOT / "experiment" / "data" / "multirepData" / "<tarfile>"
SESSION_PATH = None

_data_dir = REPO_ROOT / "experiment" / "data" / "multirepData"

if SESSION_PATH is None:
    _tarballs = sorted(_data_dir.glob("*.tar.gz"), key=lambda p: p.stat().st_mtime)
    if not _tarballs:
        raise FileNotFoundError(f"No .tar.gz files found in {_data_dir}")
    SESSION_PATH = _tarballs[-1]
    print(f"Auto-selected: {SESSION_PATH.name}")

path = Path(SESSION_PATH)
if path.suffixes[-2:] == [".tar", ".gz"]:
    session = load_session_from_tarball(path)
else:
    session = load_session(path)

print(f"Session:    {session.session_id}")
print(f"Preset:     {session.preset_name}")
print(f"Timestamp:  {session.session_timestamp}")
print(f"Tasks:      {session.n_tasks}")
print(f"Datasets:   {session.datasets}")
print(f"Rep rows:   {len(session.reputation_timeline)}")
print(f"Acc rows:   {len(session.global_accuracy)}")

Auto-selected: fast-test-local_20260603_120448.tar.gz
Session:    50360575-5aa6-4f9b-b905-c0b1f278a821
Preset:     fast-test-local
Timestamp:  20260603_120448
Tasks:      10
Datasets:   ['mnist', 'cifar-10', 'mnist', 'cifar-10', 'mnist', 'cifar-10', 'mnist', 'cifar-10', 'mnist', 'cifar-10']
Rep rows:   120
Acc rows:   40


## Access top-level data

In [3]:
rep     = session.reputation_timeline   # one row per (task, user)
acc     = session.global_accuracy       # one row per (task, round)
tasks   = session.tasks                 # list of per-task metadata dicts
preset  = session.preset

## Reputation timeline
One row per `(task_index, user)`. Contains pre/post reputation values, selection decision, and confidence.

**Key columns:** `task_index`, `dataset`, `task_type`, `user_name`, `behavior`, `was_selected`, `was_cached`, `tr_pre`, `tr_post`, `tr_all_post`, `gir_pre`, `gir_post`, `q_pre`, `q_post`, `balance_pre`, `balance_post`, `selection_score`, `confidence`, `k`

In [4]:
rep

,task_index,dataset,task_type,fingerprint,user_name,user_address,guid,behavior,was_selected,was_cached,...,tr_all_post,gir_post,q_post,q_all_post,balance_post,total_contrib_post,confidence,k,running_c_mean,m2
0,0,mnist,5,44c360e506610e5e3961cc8b39fe0ffac98f2c3114d1b7...,Balanced A,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,296baa1b-ecd2-e910-43b2-ed0f556c8caf,honest,True,False,...,{5: 0.0},0.000000,0.416667,{5: 0.4166666666666667},0.000000,0.0,0.000000,0,0.000000,0.000000
1,0,mnist,5,44c360e506610e5e3961cc8b39fe0ffac98f2c3114d1b7...,Balanced B,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,607f92d4-7776-41d1-9fd8-d1544b28d35e,honest,True,False,...,{5: 0.0},0.000000,0.416667,{5: 0.4166666666666667},0.000000,0.0,0.000000,0,0.000000,0.000000
2,0,mnist,5,44c360e506610e5e3961cc8b39fe0ffac98f2c3114d1b7...,MNIST Expert 1,0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC,bee79562-79c0-d807-1571-44d0b752438e,honest,True,False,...,{5: 0.0},0.000000,0.416667,{5: 0.4166666666666667},0.000000,0.0,0.000000,0,0.000000,0.000000
3,0,mnist,5,44c360e506610e5e3961cc8b39fe0ffac98f2c3114d1b7...,MNIST Expert 2,0x90F79bf6EB2c4f870365E785982E1f101E93b906,ac193986-b823-1eb7-af58-2839a4c10301,honest,False,False,...,{5: 0.0},0.005556,0.000000,{5: 0.0},0.000000,0.0,0.166667,1,0.000000,0.000000
4,0,mnist,5,44c360e506610e5e3961cc8b39fe0ffac98f2c3114d1b7...,CIFAR Expert 1,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,627ce64d-fccf-0d4b-3c15-a6105eefa35a,honest,False,False,...,{5: 0.0},0.005556,0.000000,{5: 0.0},0.000000,0.0,0.166667,1,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,9,cifar-10,6,8fa3dda58bc88ab42c8733ce1e7cb3d55b427515e980e8...,Cross Mal-MNIST Honest-CIFAR,0x14dC79964da2C08b23698B3D3cc7Ca32193d9955,b7fd4edd-361d-6858-0a0e-37ce12be516c,malicious,False,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.014222,0.166667,"{5: 0.16666666666666666, 1: 0.0, 2: 0.0, 3: 0....",0.000000,0.0,0.284975,2,0.324995,0.000130
116,9,cifar-10,6,8fa3dda58bc88ab42c8733ce1e7cb3d55b427515e980e8...,Malicious Both,0x23618e81E3f5cdF7f54C3d65f7FBc0aBf5B21E8f,31c4d68b-1f18-071e-dd3b-499b715548f9,malicious,False,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.014222,0.166667,"{5: 0.16666666666666666, 1: 0.0, 2: 0.0, 3: 0....",0.000000,0.0,0.284975,2,0.324995,0.000130
117,9,cifar-10,6,8fa3dda58bc88ab42c8733ce1e7cb3d55b427515e980e8...,Free-rider Both,0xa0Ee7A142d267C1f36714E4a8F75612F20a79720,34eac3e3-3c1c-9ae8-841a-94fcb2cf42e5,freerider,False,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.072000,0.333333,"{5: 0.9166666666666666, 1: 0.0, 2: 0.0, 3: 0.0...",0.000000,0.0,0.282951,2,0.328643,0.000488
118,9,cifar-10,6,8fa3dda58bc88ab42c8733ce1e7cb3d55b427515e980e8...,Passive A,0xBcd4042DE499D14e55001CcbB24a551F3b954096,520d0604-83ca-5769-ec11-2bcb5483c0b2,honest,False,False,...,"{5: 0.021939022342775793, 1: 0.0, 2: 0.0, 3: 0...",0.200044,0.916667,"{5: 0.5, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.688577,0.0,0.120178,2,0.868783,0.068871


## Global accuracy
One row per `(task_index, round)`. Aggregated from per-task run data.

**Key columns:** `task_index`, `dataset`, `round`, `objective_global_accuracy`, `objective_global_loss`, `reward_pool`, `punishment_pool`

In [5]:
acc

,round,round_time,objective_global_accuracy,objective_global_loss,reward_pool,punishment_pool,task_index,dataset
0,0,0.000000,0.0869,2.304688,1000000000000000000,0,0,mnist
1,1,1.904169,0.5375,2.240234,666666666666666667,666666666666666664,0,mnist
2,2,1.760517,0.6971,2.031250,333333333333333334,444444444444444444,0,mnist
3,3,1.774985,0.7705,1.526367,1,666666666666666669,0,mnist
4,0,0.000000,0.0940,2.304688,1000000000000000000,0,1,cifar-10
5,1,2.379325,0.1000,2.302734,666666666666666667,0,1,cifar-10
6,2,2.368279,0.1001,2.302734,333333333333333334,0,1,cifar-10
7,3,2.250037,0.1000,2.302734,1,0,1,cifar-10
8,0,0.000000,0.1182,2.302734,1000000000000000000,0,2,mnist
9,1,3.194693,0.1009,2.302734,666666666666666667,0,2,mnist


## Per-task run data
Each task in `session.tasks` may embed the full tables from its individual `.pkl` run.
Use `session.get_task_run_data(task_index)` to retrieve them, or iterate with `session.iter_run_data()`.

**Tables per task:** `global`, `users`, `votes`, `receipts`, `contributions`, `warnings`, `task_rep_calc`, `trs`

In [6]:
import pandas as pd

rows = []
for t in tasks:
    rd = t.get("run_data")
    has_data = rd is not None and any(
        (hasattr(df, "empty") and not df.empty) for df in rd.values()
    )
    rows.append({
        "task_index": t["task_index"],
        "dataset":    t["dataset"],
        "task_type":  t["task_type"],
        "was_cached": t["was_cached"],
        "has_run_data": has_data,
    })

pd.DataFrame(rows)

,task_index,dataset,task_type,was_cached,has_run_data
0,0,mnist,5,False,True
1,1,cifar-10,6,False,True
2,2,mnist,5,False,True
3,3,cifar-10,6,False,True
4,4,mnist,5,False,True
5,5,cifar-10,6,False,True
6,6,mnist,5,False,True
7,7,cifar-10,6,False,True
8,8,mnist,5,False,True
9,9,cifar-10,6,False,True


In [7]:
# Inspect a specific task
TASK_INDEX = 1

rd = session.get_task_run_data(TASK_INDEX)
if rd is None:
    print(f"Task {TASK_INDEX} has no embedded run data.")
else:
    global_df       = rd.get("global",        pd.DataFrame())
    users_df        = rd.get("users",          pd.DataFrame())
    votes_df        = rd.get("votes",          pd.DataFrame())
    receipts_df     = rd.get("receipts",       pd.DataFrame())
    contributions_df= rd.get("contributions",  pd.DataFrame())
    task_rep_df     = rd.get("task_rep_calc",  pd.DataFrame())
    warnings_df     = rd.get("warnings",       pd.DataFrame())
    trs_df          = rd.get("trs",            pd.DataFrame())

    for name, df in [("global", global_df), ("users", users_df), ("votes", votes_df),
                     ("receipts", receipts_df), ("contributions", contributions_df),
                     ("task_rep_calc", task_rep_df), ("warnings", warnings_df), ("trs", trs_df)]:
        print(f"{name}: {df.shape}")

global: (4, 7)
users: (20, 17)
votes: (60, 11)
receipts: (68, 5)
contributions: (15, 17)
task_rep_calc: (5, 13)
warnings: (0, 0)
trs: (1, 5)


In [8]:
global_df

,experiment_id,round,round_time,objective_global_accuracy,objective_global_loss,reward_pool,punishment_pool
0,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,0,0.000000,0.0940,2.304688,1000000000000000000,0
1,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,2.379325,0.1000,2.302734,666666666666666667,0
2,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,2,2.368279,0.1001,2.302734,333333333333333334,0
3,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,3,2.250037,0.1000,2.302734,1,0


In [9]:
users_df

,experiment_id,round,user_number,state,behavior,role,grs,address,id,subjective_personal_accuracy,subjective_personal_loss,subjective_global_accuracy,subjective_global_loss,round_reputation_assigned,reward_delta,is_reward,merged
0,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,0,0,active,Attitude.Honest,Attitude.Honest,1000000000000000000,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,e032a3fa-3ae0-485c-9ebf-ea46b8dbe881,NaN,NaN,NaN,NaN,NaN,NaN,None,None
1,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,0,11,active,Attitude.Honest,Attitude.Honest,1000000000000000000,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,7030b401-5437-4b18-9e82-0eb8b44cfe29,NaN,NaN,NaN,NaN,NaN,NaN,None,None
2,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,0,2,active,Attitude.Honest,Attitude.Honest,1000000000000000000,0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC,e07fdd25-ef16-439d-8744-3ee37ed790d6,NaN,NaN,NaN,NaN,NaN,NaN,None,None
3,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,0,1,active,Attitude.Honest,Attitude.Honest,1000000000000000000,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,51fbf926-e516-4966-9a97-007df2ee0a50,NaN,NaN,NaN,NaN,NaN,NaN,None,None
4,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,0,9,active,Attitude.Honest,Attitude.FreeRider,1000000000000000000,0xa0Ee7A142d267C1f36714E4a8F75612F20a79720,21ceb7cc-6b5e-4d9e-b21b-47776007fe49,NaN,NaN,NaN,NaN,NaN,NaN,None,None
5,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,0,active,Attitude.Honest,Attitude.Honest,1057511428392509126,None,e032a3fa-3ae0-485c-9ebf-ea46b8dbe881,0.100000,2.302578,0.1000,2.302734,1.000000e+18,5.751143e+16,True,True
6,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,11,active,Attitude.Honest,Attitude.Honest,1129400713883149618,None,7030b401-5437-4b18-9e82-0eb8b44cfe29,0.090000,2.304531,0.0987,2.304688,1.000000e+18,1.294007e+17,True,True
7,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,2,active,Attitude.Honest,Attitude.Honest,1013864362201765397,None,e07fdd25-ef16-439d-8744-3ee37ed790d6,0.203333,2.170156,0.1221,2.314453,3.000000e+18,1.386436e+16,True,True
8,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,1,active,Attitude.Honest,Attitude.Honest,1129400713883149618,None,51fbf926-e516-4966-9a97-007df2ee0a50,0.100000,2.300000,0.1040,2.300781,1.000000e+18,1.294007e+17,True,True
9,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,9,active,Attitude.FreeRider,Attitude.FreeRider,1003156114972759571,None,21ceb7cc-6b5e-4d9e-b21b-47776007fe49,0.000000,0.000000,0.1016,2.304688,3.000000e+18,3.156115e+15,True,True


In [10]:
votes_df

,experiment_id,round,giver_id,receiver_id,giver_address,receiver_address,vote_feedback_score,vote_prev_accuracy,vote_prev_loss,vote_accuracy,vote_loss
0,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,e032a3fa-3ae0-485c-9ebf-ea46b8dbe881,7030b401-5437-4b18-9e82-0eb8b44cfe29,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,1,775,230,1025,230
1,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,e032a3fa-3ae0-485c-9ebf-ea46b8dbe881,e07fdd25-ef16-439d-8744-3ee37ed790d6,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC,1,775,230,1175,232
2,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,e032a3fa-3ae0-485c-9ebf-ea46b8dbe881,51fbf926-e516-4966-9a97-007df2ee0a50,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,1,775,230,1025,230
3,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,e032a3fa-3ae0-485c-9ebf-ea46b8dbe881,21ceb7cc-6b5e-4d9e-b21b-47776007fe49,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,0xa0Ee7A142d267C1f36714E4a8F75612F20a79720,1,775,230,1125,238
4,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,7030b401-5437-4b18-9e82-0eb8b44cfe29,e032a3fa-3ae0-485c-9ebf-ea46b8dbe881,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,1,900,230,1000,230
5,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,7030b401-5437-4b18-9e82-0eb8b44cfe29,e07fdd25-ef16-439d-8744-3ee37ed790d6,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC,1,900,230,1000,231
6,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,7030b401-5437-4b18-9e82-0eb8b44cfe29,51fbf926-e516-4966-9a97-007df2ee0a50,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,1,900,230,1500,230
7,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,7030b401-5437-4b18-9e82-0eb8b44cfe29,21ceb7cc-6b5e-4d9e-b21b-47776007fe49,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,0xa0Ee7A142d267C1f36714E4a8F75612F20a79720,1,900,230,800,241
8,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,e07fdd25-ef16-439d-8744-3ee37ed790d6,e032a3fa-3ae0-485c-9ebf-ea46b8dbe881,0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,-1,133,229,0,231
9,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,e07fdd25-ef16-439d-8744-3ee37ed790d6,7030b401-5437-4b18-9e82-0eb8b44cfe29,0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,-1,133,229,133,233


In [11]:
receipts_df

,experiment_id,round,tx_type,tx_hash,gas_used
0,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,slot,2900237ed5b17e2f0d2e0574202700c06a2bd24275f4f5...,50837
1,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,slot,dad3b6cd616b936b23d0dbbdf45f206cff21796a9fd5d7...,50825
2,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,slot,339fda1fb60dd2b132cedc3f190112d659f828bfb4faeb...,50837
3,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,slot,86d8aae765d07a2bbf7455f555df7990f9e700e7e7f4df...,50837
4,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,slot,803c498affaa7da27df3957083e205cd9acfacda6467e1...,50837
...,...,...,...,...,...
63,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,4,exit,ae268a52005c888583775e9257d76057367bf52c8fff29...,56148
64,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,4,exit,1bc314889b35d0fc80bf6bc415f108383d8d0f296ab555...,58712
65,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,4,exit,9937d6ec4e94ca8a21ad595a117b13209e2b3d1f2a1974...,56476
66,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,4,exit,4493e79717485a088a3a7bfe53be5d61777bf159295e8f...,53912


In [12]:
contributions_df

,experiment_id,round,user_id,user_address,contribution_score,user_mad_avg,current_excluded_values,current_accepted_values,current_mad_median,current_mad_value,current_mad_max_deviation,prev_avg_round_val_after_mad,previous_excluded_values,previous_accepted_values,previous_mad_median,previous_mad_value,previous_mad_max_deviation
0,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,0,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,172534285177527391,230.250000,[],"[230, 231, 230, 230]",230.0,0.0,NaN,229.8,[],"[230, 230, 229, 230, 230]",230.0,0.0,None
1,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,11,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,388202141649448884,230.000000,[233],"[230, 230, 230]",230.0,0.0,NaN,229.8,[],"[230, 230, 229, 230, 230]",230.0,0.0,None
2,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,2,0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC,41593086605296196,231.666667,[230],"[232, 231, 232]",231.5,0.5,0.815419,229.8,[],"[230, 230, 229, 230, 230]",230.0,0.0,None
3,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,1,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,388202141649448884,230.000000,[232],"[230, 230, 230]",230.0,0.0,NaN,229.8,[],"[230, 230, 229, 230, 230]",230.0,0.0,None
4,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,1,9,0xa0Ee7A142d267C1f36714E4a8F75612F20a79720,9468344918278715,238.000000,"[241, 235]","[238, 238]",238.0,1.5,2.446256,229.8,[],"[230, 230, 229, 230, 230]",230.0,0.0,None
5,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,2,0,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,500000000000000000,230.000000,[],"[230, 230, 230, 230]",230.0,0.0,NaN,230.0,[],"[230, 230, 230, 230, 230]",230.0,0.0,None
6,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,2,11,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,500000000000000000,230.000000,[],"[230, 230, 230, 230]",230.0,0.0,NaN,230.0,[],"[230, 230, 230, 230, 230]",230.0,0.0,None
7,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,2,2,0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC,500000000000000000,230.000000,[],"[230, 230, 230, 230]",230.0,0.0,NaN,230.0,[],"[230, 230, 230, 230, 230]",230.0,0.0,None
8,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,2,1,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,500000000000000000,230.000000,[],"[230, 230, 230, 230]",230.0,0.0,NaN,230.0,[],"[230, 230, 230, 230, 230]",230.0,0.0,None
9,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,2,9,0xa0Ee7A142d267C1f36714E4a8F75612F20a79720,-1000000000000000000,231.000000,"[232, 229]","[231, 231]",231.0,0.5,0.815419,230.0,[],"[230, 230, 230, 230, 230]",230.0,0.0,None


In [13]:
import pandas as pd

all_contributions = pd.concat([
    rd.get("contributions", pd.DataFrame()).assign(task_index=task_index)
    for task_index, dataset, rd in session.iter_run_data()
    if rd.get("contributions") is not None and not rd["contributions"].empty
], ignore_index=True)

all_contributions["contribution_score"] /= 1e18

user_names = rep[["user_address", "user_name"]].drop_duplicates()
all_contributions = all_contributions.merge(user_names, on="user_address", how="left")
all_contributions["user"] = all_contributions["user_name"].fillna(all_contributions["user_id"].astype(str))

pivot = all_contributions.pivot_table(
    index="user",
    columns="task_index",
    values="contribution_score",
    aggfunc="mean"
).rename_axis(None, axis=1)

selected_counts = all_contributions.groupby("task_index")["user"].nunique().rename("# selected")
pivot.loc["# selected"] = selected_counts
pivot

/tmp/ipykernel_87060/509570012.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_contributions = pd.concat([


,0,1,2,3,4,5,7,9
user,,,,,,,,
Balanced A,NaN,0.453808,0.632850,0.558908,0.309596,1.038158,NaN,NaN
Balanced B,NaN,0.525697,NaN,NaN,0.339899,NaN,NaN,NaN
CIFAR Expert 1,NaN,NaN,NaN,0.054598,NaN,0.087719,0.250022,NaN
CIFAR Expert 2,NaN,NaN,NaN,0.804598,NaN,-0.164035,NaN,NaN
Cross Honest-MNIST Mal-CIFAR,0.953795,NaN,0.584541,NaN,0.267172,NaN,NaN,NaN
Cross Mal-MNIST Honest-CIFAR,NaN,NaN,NaN,NaN,NaN,NaN,0.125008,0.4
Free-rider Both,NaN,-0.619066,NaN,NaN,NaN,NaN,NaN,0.2
MNIST Expert 1,NaN,0.113864,NaN,-1.000000,NaN,NaN,NaN,NaN
MNIST Expert 2,NaN,NaN,-1.000000,NaN,NaN,-1.000000,0.249951,NaN


In [14]:
task_rep_df

,experiment_id,user_id,address,task_type,k,running_c_mean,m2,global_task_rep,global_integrity_rep,task_rep_delta,final_grs,positive_votes,total_votes
0,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,e032a3fa-3ae0-485c-9ebf-ea46b8dbe881,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,6,0,0.894254,0.0,0.029808,0.088889,251955872836953570,1426192228925019276,8,12
1,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,7030b401-5437-4b18-9e82-0eb8b44cfe29,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,6,0,0.945604,0.0,0.031520,0.088889,323845158327594062,1498081514415659768,8,12
2,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,e07fdd25-ef16-439d-8744-3ee37ed790d6,0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC,6,0,0.731465,0.0,0.024382,0.112500,24051182499429640,1102074435886200146,9,12
3,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,51fbf926-e516-4966-9a97-007df2ee0a50,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,6,0,0.945604,0.0,0.031520,0.088889,323845158327594062,1498081514415659768,8,12
4,cifar-10-loss_only-1-0.1-1-1.0-False-{a325556e...,21ceb7cc-6b5e-4d9e-b21b-47776007fe49,0xa0Ee7A142d267C1f36714E4a8F75612F20a79720,6,0,0.339693,0.0,0.011323,0.138889,-524429693642538963,475570306357461037,10,12


In [15]:
import pandas as pd

all_task_rep = pd.concat([
    rd.get("task_rep_calc", pd.DataFrame()).assign(task_index=task_index)
    for task_index, dataset, rd in session.iter_run_data()
    if rd.get("task_rep_calc") is not None and not rd["task_rep_calc"].empty
], ignore_index=True)

all_task_rep["task_rep_delta"] = pd.to_numeric(all_task_rep["task_rep_delta"], errors="coerce") / 1e18

user_names = rep[["user_address", "user_name"]].drop_duplicates()
all_task_rep = all_task_rep.merge(user_names, left_on="address", right_on="user_address", how="left")
all_task_rep["user"] = all_task_rep["user_name"].fillna(all_task_rep["user_id"].astype(str))

pivot = all_task_rep.pivot_table(
    index="user",
    columns="task_index",
    values="task_rep_delta",
    aggfunc="mean"
).rename_axis(None, axis=1)

selected_counts = all_task_rep.groupby("task_index")["user"].nunique().rename("# selected")
pivot.loc["# selected"] = selected_counts
pivot

,0,1,2,3,4,5,6,7,8,9
user,,,,,,,,,,
Balanced A,NaN,0.251956,0.316425,0.258621,0.309596,0.464912,-1.0,NaN,NaN,NaN
Balanced B,NaN,0.323845,NaN,NaN,0.339899,NaN,-1.0,NaN,-1.0,-1.000000
CIFAR Expert 1,-1.000000,NaN,NaN,-0.020115,NaN,0.043860,NaN,-0.518515,-1.0,NaN
CIFAR Expert 2,-1.000000,NaN,NaN,0.402299,NaN,-0.184649,NaN,NaN,NaN,NaN
Cross Honest-MNIST Mal-CIFAR,0.953795,NaN,0.292271,NaN,0.267172,NaN,NaN,NaN,NaN,NaN
Cross Mal-MNIST Honest-CIFAR,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,-0.537036,-1.0,-0.496296
Free-rider Both,NaN,-0.524430,NaN,NaN,-1.000000,NaN,-1.0,NaN,NaN,-0.525926
MNIST Expert 1,NaN,0.024051,NaN,-0.703704,NaN,NaN,NaN,NaN,-1.0,NaN
MNIST Expert 2,-1.000000,NaN,-0.703704,NaN,NaN,-0.703704,NaN,-0.518526,NaN,-1.000000


## Merge users with their votes (per task)
Attach receiver behavior/role to each vote row.

In [ ]:
if votes_df.empty or users_df.empty:
    print("votes or users is empty for this task.")
else:
    votes_with_receiver = votes_df.merge(
        users_df[["experiment_id", "round", "address", "behavior", "role"]],
        left_on=["experiment_id", "round", "receiver_address"],
        right_on=["experiment_id", "round", "address"],
        how="left"
    ).rename(columns={
        "behavior": "receiver_behavior",
        "role":     "receiver_role",
    }).drop(columns=["address"])

    votes_with_receiver

## Reputation analysis across all tasks

In [ ]:
# Selection frequency per user
rep[rep["was_selected"]].groupby(["user_name", "behavior"])["task_index"].count().rename("times_selected").reset_index()

In [ ]:
# Mean final TR per behavior group
rep.groupby("behavior")[["tr_post", "gir_post", "q_post", "balance_post"]].mean().round(4)

In [ ]:
# Filter reputation timeline to a specific task
rep[rep["task_index"] == TASK_INDEX]

## Jupyter notes

Display all columns:

```python
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
```

Iterate all tasks with run data:

```python
for task_index, dataset, rd in session.iter_run_data():
    global_df = rd.get("global", pd.DataFrame())
    ...
```